# **Atelier Préparation de Données Textuelles**

### **Contexte** 
Une entreprise souhaite développer un système capable de classer automatiquement des avis 
clients afin d'identifier leur sentiment. 
Les avis proviennent de plusieurs sources : site web, application mobile, formulaire de satisfaction, 
réseaux sociaux et service client. 
Les données collectées sont cependant de qualité variable : textes vides, doublons, fautes de frappe, 
majuscules/minuscules, caractères spéciaux, emojis, URLs, mentions, répétitions de caractères, 
textes très courts ou très longs, plusieurs catégories de sentiment et quelques valeurs manquantes. 
Avant de construire un modèle de Machine Learning ou de Deep Learning, les apprenants doivent 
donc construire un pipeline complet de préparation des données textuelles. 

---
## **Partie 1 – Exploration du corpus**

### **1) Charger les données CSV**

In [46]:
import pandas as pd

df = pd.read_csv("../data/smart_reviews_raw.csv")
df.head()

,id_avis,date,source,produit,texte,sentiment,note,langue
0,AV0001,2026-02-27,mobile,Ordinateur NovaBook,"Très bonne expérience, simple et efficace.",positif,4,fr
1,AV0002,2026-01-09,web,SmartPhone X,Très satisfait de mon achat 👍 #avis,positif,4,fr
2,AV0003,2026-07-03,réseaux_sociaux,Écouteurs AirSound,"Produit parfait, rien à signaler.",positif,4,fr
3,AV0004,2026-06-28,sav,SmartWatch Pro,"Produit excellent, je suis très satisfait. !!!",positif,5,fr
4,AV0005,2026-01-24,sav,SmartPhone X,LA BATTERIE TIENT VRAIMENT BIEN ET L'ÉCRAN EST...,positif,5,fr


### **2) Combien d'avis contient le dataset ?**

In [32]:
print(f"Le dataset contient {len(df)} avis.")

Le dataset contient 1200 avis.


### **3) Combien de colonnes possède-t-il ?**

In [33]:
print(f"Le dataset possède {df.shape[1]} colonnes.")
print("Colonnes :", df.columns.tolist())

Le dataset possède 8 colonnes.
Colonnes : ['id_avis', 'date', 'source', 'produit', 'texte', 'sentiment', 'note', 'langue']


### **4) Quel est le type de chaque colonne ?**

In [34]:
print(df.dtypes)

id_avis        str
date           str
source         str
produit        str
texte          str
sentiment      str
note         int64
langue         str
dtype: object


### **5) Existe-t-il des valeurs manquantes ?**

In [35]:
print("Valeurs manquantes par colonne :")
print(df.isnull().sum())
print("\nTotal de valeurs manquantes :", df.isnull().sum().sum())

Valeurs manquantes par colonne :
id_avis      0
date         0
source       0
produit      0
texte        5
sentiment    0
note         0
langue       0
dtype: int64

Total de valeurs manquantes : 5


### **6) Identifier quelques types de texte**

In [36]:
# Texte normal
print("=== Texte normal ===")
print(df.loc[0, "texte"])

=== Texte normal ===
Très  bonne  expérience,  simple  et  efficace.


In [37]:
# Texte vide / NaN
print("\n=== Texte vide / NaN ===")
print(df[df["texte"].isna() | (df["texte"].astype(str).str.strip() == "")]["texte"].head(3))



=== Texte vide / NaN ===
46     NaN
108    NaN
310    NaN
Name: texte, dtype: str


In [38]:
# Texte contenant une URL
print("\n=== Texte avec URL ===")
print(df[df["texte"].astype(str).str.contains(r"https?://", na=False)]["texte"].head(2).tolist())



=== Texte avec URL ===
['Produit parfait, rien à signaler. https://example.com/commande/17', 'Très bonne expérience, simple et efficace. https://example.com/commande/31']


In [39]:
# Texte contenant une mention
print("\n=== Texte avec mention ===")
print(df[df["texte"].astype(str).str.contains(r"@\w+", na=False)]["texte"].head(2).tolist())


=== Texte avec mention ===
['@client Livraison rapide et produit conforme à mes attentes.', '@client Service client réactif et commande reçue rapidement.']


In [40]:
# Texte contenant un hashtag
print("\n=== Texte avec hashtag ===")
print(df[df["texte"].astype(str).str.contains(r"#\w+", na=False)]["texte"].head(2).tolist())



=== Texte avec hashtag ===
['Très satisfait de mon achat 👍 #avis', "La batterie tient vraiment bien et l'écran est superbe. #avis"]


In [41]:
# Texte contenant des emojis
print("\n=== Texte avec emojis ===")
print(df[df["texte"].astype(str).str.contains(r"[👍😊😡]", na=False)]["texte"].head(2).tolist())



=== Texte avec emojis ===
['Très satisfait de mon achat 👍 #avis', 'Très satisfait de mon achat 👍 😊']


In [42]:
# Texte avec beaucoup de ponctuation
print("\n=== Texte avec beaucoup de ponctuation ===")
print(df[df["texte"].astype(str).str.contains(r"!!!", na=False)]["texte"].head(2).tolist())



=== Texte avec beaucoup de ponctuation ===
['Produit excellent, je suis très satisfait. !!!', 'Très satisfait de mon achat 👍 !!!']


In [43]:
# Texte en majuscules
print("\n=== Texte en majuscules ===")
print(df[df["texte"].astype(str).str.isupper()]["texte"].head(2).tolist())



=== Texte en majuscules ===
["LA BATTERIE TIENT VRAIMENT BIEN ET L'ÉCRAN EST SUPERBE.", 'SERVICE CLIENT RÉACTIF ET COMMANDE REÇUE RAPIDEMENT.']


In [44]:
# Texte avec répétition de caractères
print("\n=== Texte avec répétition de caractères ===")
print(df[df["texte"].astype(str).str.contains(r"(.)\1{2,}", na=False)]["texte"].head(2).tolist())


=== Texte avec répétition de caractères ===
['Produit excellent, je suis très satisfait. !!!', 'Très satisfait de mon achat 👍 !!!']


C:\Users\HP\AppData\Local\Temp\ipykernel_21388\1618673901.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print(df[df["texte"].astype(str).str.contains(r"(.)\1{2,}", na=False)]["texte"].head(2).tolist())


### **7) Mesurer la longueur des textes**

In [49]:
# Remplacer les NaN par chaîne vide
df["texte"] = df["texte"].fillna("")

# Calculer la longueur en évitant les erreurs
df["longueur"] = df["texte"].astype(str).apply(lambda x: len(x.strip()))
print("Longueur minimale :", df["longueur"].min())
print("Longueur maximale :", df["longueur"].max())
print("Longueur moyenne  :", round(df["longueur"].mean(), 2))
print("Médiane           :", df["longueur"].median())
print("\nQuartiles :")
print(df["longueur"].quantile([0.25, 0.5, 0.75]))


Longueur minimale : 0
Longueur maximale : 635
Longueur moyenne  : 48.96
Médiane           : 48.0

Quartiles :
0.25    37.0
0.50    48.0
0.75    56.0
Name: longueur, dtype: float64


### **8) Visualiser la distribution de la longueur des avis**

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(df["longueur"], bins=40, edgecolor="black", color="steelblue")
plt.title("Distribution de la longueur des avis")
plt.xlabel("Longueur (nombre de caractères)")
plt.ylabel("Nombre d'avis")
plt.axvline(df["longueur"].mean(), color="red", linestyle="--", label=f"Moyenne = {df['longueur'].mean():.1f}")
plt.legend()
plt.show()